# Similar emergencies

For a new emergency, EMRIS finds the five most similar past emergencies and suggests the actions that were taken in at least two of them. Similarity uses the event type, the place, the month and, when available, the recorded impact, with a nearest-neighbors search.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))
DATA = ROOT / "data"

## Train and evaluate

`train_similarity` builds the search index and measures precision at 5: the share of neighbors with the same event type and the same impact level as the query. It is compared with a baseline that only knows the event type.

In [ ]:
import pandas as pd

from ml.actions import ACTION_VOCABULARY
from ml.data import load_frame
from ml.similarity import suggest_actions, train_similarity

frame = load_frame(DATA / "ungrd_emergencies_v2.csv", DATA / "divipola_dane.csv")
result = train_similarity(frame)
metrics = result.metrics
pd.Series(
    {
        "Precision at 5, impact known": metrics["precision_at_5"],
        "Precision at 5, impact unknown": metrics["precision_at_5_without_impact"],
        "Baseline (same event type only)": metrics["baseline_precision_at_5"],
    }
).round(3)

## An October flood

One flood from October is used as a new report. It is part of the history, so it is excluded from its own neighbors. The table lists its five neighbors, and the actions suggested from them follow with their labels.

In [ ]:
model = result.model
months = pd.to_datetime(frame["occurred_on"]).dt.month
floods = frame[(frame["event_type"] == "Flood") & (months == 10) & (frame["municipality"] != "")]
query = floods.head(1)
row = query.iloc[0]
print(f"Query: {row['event_type']} on {row['occurred_on']}, municipality {row['municipality']}")

neighbors = model.neighbors(query)[0]
print(pd.DataFrame(
    {
        "reference": [n.reference for n in neighbors],
        "event type": [n.event_type for n in neighbors],
        "date": [n.occurred_on for n in neighbors],
        "distance": [round(n.distance, 3) for n in neighbors],
    }
).to_string(index=False))

print("Suggested actions:")
for code in suggest_actions(neighbors):
    print(f"  - {ACTION_VOCABULARY[code].label}")